<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/05.%20Video%20Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Video Annotation**
> MARC - Multidisciplinary AI Research Centre | UOP


This notebook serves as the command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)


## 📚 Initial Setup & Configuration


### 🔧 One-Time Setup Instructions

#### **Step 1: Get Access to the Shared Dataset Folder**
1. Ask your **admin** to share the Google Drive folder with you
2. You'll receive an email with a "Shared with you" notification
3. Open the shared folder in Google Drive
4. The folder should be named: **`Wearable-Biomechanics`**

#### **Step 2: Add Shortcut to Your Drive**
1. In Google Drive, locate the shared folder under **"Shared with me"**
2. **Right-click** on the folder named `Wearable-Biomechanics`
3. Select **"Add shortcut to Drive"** (or **"Organize → Add shortcut"**)
4. Navigate to your **`My Drive > Datasets`** folder
   - If `Datasets` folder doesn't exist, create it first
5. Click **"Add Shortcut"**

#### **Step 3: Verify the Setup**
- Go to **`My Drive > Datasets`** in your Google Drive
- You should see **`Wearable-Biomechanics`** with a shortcut icon ⤴️
- The full path will be: **`MyDrive/Datasets/Wearable-Biomechanics`**


### ⚠️ Important Notes
- 🔑 **Permissions**: You need at least **"Editor"** access to upload data
- 🔗 **Shortcuts**: Always use the shortcut in your Drive, not the original shared folder
- 📂 **Structure**: The system will auto-create `/index`, `/raw_data`, and `/processed_data` subfolders
- 🚫 **Don't delete** the shortcut or you'll lose access to shared data


### 📁 Folder Structure After Setup
```
My Drive/
└── Datasets/
    └── Wearable-Biomechanics/
        ├── index/
        │   ├── entities_subjects.json
        │   └── entries_experiments.json
        ├── raw_data/
        │   └── experiments/
        └── related_files/
        │    └── experiments/
        └── processed_data/
            └── experiments/
```


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local", # Use "local" since you are mounting Google Drive via Colab directly
    entity_id_format="sub_{num:04d}", # You can drop the `f""` string formatting here now
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [ ]:
# @title 🎬 Frame-by-Frame Video Player (fast) { display-mode: "form" }
import os, time, cv2, socket, threading, subprocess
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from urllib.parse import unquote, urlparse
from IPython.display import display, HTML

entry_type      = "fault_tolerant_trials"            # @param ["complete_trials", "fault_tolerant_trials"]
target_entry_id = "fault_tolerant_trials_0007"       # @param {type:"string"}
proxy_height    = 720                          # @param [480, 540, 720, 1080] {type:"raw"}
quality_crf     = 22
force_reencode  = False

# ── 1. Download source video ───────────────────────────────────────────
video_dir = "/tmp/video_player"
os.makedirs(video_dir, exist_ok=True)
src = store.download_related_file(
    entry_type=entry_type, entry_id=target_entry_id,
    label="synced_video", dest_folder=video_dir,
)
if not src or not os.path.exists(src):
    raise FileNotFoundError(f"❌ No synced_video for {target_entry_id}")

# ── 2. Encode lightweight, seek-friendly proxy (cached) ────────────────
proxy_name = f"proxy_{target_entry_id}_{proxy_height}p_crf{quality_crf}.mp4"
proxy_path = os.path.join(video_dir, proxy_name)

if force_reencode or not os.path.exists(proxy_path) \
        or os.path.getmtime(proxy_path) < os.path.getmtime(src):
    cap = cv2.VideoCapture(src)
    src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    vf = ["-vf", f"scale=-2:{proxy_height}"] if src_h > proxy_height else []
    print(f"⚡ Encoding {proxy_height}p proxy (one-time)...")
    t0 = time.time()
    subprocess.run(
        ["ffmpeg", "-y", "-i", src, *vf,
         "-c:v", "libx264", "-preset", "veryfast",
         "-crf", str(quality_crf),
         "-g", "6", "-bf", "0",          # short GOP, no B-frames → instant exact seeks
         "-threads", "0",
         "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an",
         proxy_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print(f"✅ Done in {time.time()-t0:.1f}s · {os.path.getsize(proxy_path)/1e6:.1f} MB")
else:
    print("♻️ Using cached proxy")

cap = cv2.VideoCapture(proxy_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# ── 3. HTTP server WITH Range support (required for fast seeking) ──────
class RangeHandler(BaseHTTPRequestHandler):
    def log_message(self, *a): pass

    def _serve(self, head=False):
        name = os.path.basename(unquote(urlparse(self.path).path))
        path = os.path.join(video_dir, name)
        if not os.path.isfile(path):
            self.send_error(404); return
        size = os.path.getsize(path)
        start, end = 0, size - 1
        rng = self.headers.get("Range")
        status = 200
        if rng and rng.startswith("bytes="):
            s, _, e = rng[6:].partition("-")
            try:
                if s == "":                       # suffix range
                    start = max(0, size - int(e))
                else:
                    start = int(s)
                    if e: end = min(int(e), size - 1)
                status = 206
            except ValueError:
                pass
            if start > end or start >= size:
                self.send_response(416)
                self.send_header("Content-Range", f"bytes */{size}")
                self.end_headers(); return
        length = end - start + 1
        self.send_response(status)
        self.send_header("Content-Type", "video/mp4")
        self.send_header("Accept-Ranges", "bytes")
        self.send_header("Content-Length", str(length))
        self.send_header("Cache-Control", "max-age=3600")
        if status == 206:
            self.send_header("Content-Range", f"bytes {start}-{end}/{size}")
        self.end_headers()
        if head: return
        try:
            with open(path, "rb") as f:
                f.seek(start)
                left = length
                while left > 0:
                    chunk = f.read(min(1 << 20, left))
                    if not chunk: break
                    self.wfile.write(chunk)
                    left -= len(chunk)
        except (BrokenPipeError, ConnectionResetError):
            pass

    def do_GET(self):  self._serve()
    def do_HEAD(self): self._serve(head=True)

# stop any server left over from a previous run
old = globals().get("_vp_httpd")
if old:
    try: old.shutdown(); old.server_close()
    except Exception: pass

def _free_port(start=8123):
    for p in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try: s.bind(("", p)); return p
            except OSError: continue
    raise RuntimeError("no free port")

PORT = _free_port()
_vp_httpd = ThreadingHTTPServer(("", PORT), RangeHandler)
_vp_httpd.daemon_threads = True
threading.Thread(target=_vp_httpd.serve_forever, daemon=True).start()

try:
    from google.colab import output
    output.no_vertical_scroll()
    base_url = (output.eval_js(f"google.colab.kernel.proxyPort({PORT})") or "").rstrip("/")
except Exception:
    base_url = f"http://localhost:{PORT}"

video_url = f"{base_url}/{proxy_name}"

# ── 4. Minimal player UI ───────────────────────────────────────────────
HTML_TEMPLATE = r"""
<div id="root" tabindex="0" style="background:#18181b;color:#f4f4f5;padding:10px 12px;border-radius:10px;font-family:system-ui,sans-serif;max-width:800px;box-sizing:border-box;border:1px solid #3f3f46;outline:none;">

  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
    <span style="font-size:13px;font-weight:bold;color:#38bdf8;">🎥 __ENTRY_ID__</span>
    <span id="readout" style="background:#27272a;padding:4px 12px;border-radius:6px;font-size:16px;font-weight:bold;color:#facc15;border:1px solid #52525b;font-variant-numeric:tabular-nums;">Frame 0 / __TOTAL_M1__</span>
  </div>

  <div style="background:#000;border-radius:6px;overflow:hidden;">
    <video id="vid" src="__VIDEO_URL__" preload="auto" muted playsinline
           style="width:100%;max-height:70vh;display:block;cursor:pointer;"></video>
  </div>

  <input type="range" id="scrub" min="0" max="__TOTAL_M1__" value="0" step="1"
         style="width:100%;height:18px;margin-top:6px;cursor:pointer;">

  <div style="display:flex;flex-wrap:wrap;gap:6px;align-items:center;">
    <button class="b" data-d="-10">-10</button>
    <button class="b" data-d="-5">-5</button>
    <button class="b a" data-d="-1">◀ -1</button>
    <button id="play" class="b" style="background:#2563eb;font-weight:bold;min-width:84px;">▶ Play</button>
    <button class="b a" data-d="1">+1 ▶</button>
    <button class="b" data-d="5">+5</button>
    <button class="b" data-d="10">+10</button>
  </div>

  <div style="margin-top:6px;font-size:11px;color:#a1a1aa;">
    Click the player first, then: <b>Space</b> play/pause · <b>←/→</b> ±1 · <b>Shift+←/→</b> ±5 · <b>↑/↓</b> ±10
  </div>
</div>

<style>
  .b{padding:6px 12px;background:#3f3f46;color:#fff;border:none;border-radius:5px;cursor:pointer;font-size:13px;font-weight:500;touch-action:manipulation;user-select:none}
  .b:hover{filter:brightness(1.25)} .b.a{color:#facc15;font-weight:bold}
</style>

<script>
(function(){
  const FPS = __FPS__;
  let LAST  = __TOTAL_M1__;

  const root = document.getElementById("root");
  const vid  = document.getElementById("vid");
  const scrub= document.getElementById("scrub");
  const out  = document.getElementById("readout");
  const play = document.getElementById("play");

  let cur = 0;          // frame currently shown / logical frame while paused
  let target = 0;       // latest requested frame
  let sought = -1;      // frame of the seek currently in flight
  let seeking = false;

  const clamp = f => Math.max(0, Math.min(LAST, f|0));

  function show(f){
    cur = f;
    out.textContent = "Frame " + f + " / " + LAST;
    scrub.value = f;
  }

  // Seek to middle of the frame interval → always lands on the right frame
  function doSeek(){
    sought = target;
    seeking = true;
    vid.currentTime = (target + 0.5) / FPS;
    show(target);
  }
  function goto(f){
    target = clamp(f);
    if (!seeking) doSeek();       // otherwise applied when current seek finishes
  }
  vid.addEventListener("seeked", function(){
    seeking = false;
    if (target !== sought) doSeek();   // only jump to the latest request
  });

  function step(d){
    if (!vid.paused) vid.pause();
    goto((seeking ? target : cur) + d);
  }

  function toggle(){
    if (vid.paused){
      if (cur >= LAST) goto(0);
      vid.play();
    } else vid.pause();
  }

  // Exact frame readout while playing
  function onFrame(now, meta){
    if (!vid.paused) show(clamp(Math.round(meta.mediaTime * FPS)));
    vid.requestVideoFrameCallback(onFrame);
  }
  if ("requestVideoFrameCallback" in vid) vid.requestVideoFrameCallback(onFrame);
  else vid.addEventListener("timeupdate", function(){
    if (!vid.paused) show(clamp(Math.round(vid.currentTime * FPS)));
  });

  vid.addEventListener("play",  () => play.textContent = "⏸ Pause");
  vid.addEventListener("pause", () => { play.textContent = "▶ Play"; target = cur = clamp(Math.round(vid.currentTime * FPS - 0.5)); show(cur); });
  vid.addEventListener("ended", () => { play.textContent = "▶ Play"; show(LAST); });

  vid.addEventListener("loadedmetadata", function(){
    const n = Math.round(vid.duration * FPS) - 1;
    if (n > 0){ LAST = n; scrub.max = n; show(0); }
    if (window.google && google.colab && google.colab.output)
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight + 8, true);
  });

  // Controls
  play.onclick = toggle;
  vid.onclick  = () => { root.focus(); toggle(); };
  document.querySelectorAll("button[data-d]").forEach(b =>
    b.onclick = () => { root.focus(); step(parseInt(b.dataset.d)); });
  scrub.addEventListener("input", () => { if (!vid.paused) vid.pause(); goto(parseInt(scrub.value)); });

  root.addEventListener("keydown", function(e){
    const k = e.code;
    if (k === "Space")            { e.preventDefault(); toggle(); }
    else if (k === "ArrowRight")  { e.preventDefault(); step(e.shiftKey ? 5 : 1); }
    else if (k === "ArrowLeft")   { e.preventDefault(); step(e.shiftKey ? -5 : -1); }
    else if (k === "ArrowUp")     { e.preventDefault(); step(10); }
    else if (k === "ArrowDown")   { e.preventDefault(); step(-10); }
  });
  root.focus();
})();
</script>
"""

html = (HTML_TEMPLATE
        .replace("__VIDEO_URL__", video_url)
        .replace("__FPS__",       repr(float(fps)))
        .replace("__TOTAL_M1__",  str(total_frames - 1))
        .replace("__ENTRY_ID__",  target_entry_id))
display(HTML(html))

📥 Downloaded to: /tmp/video_player/complete_trials_0012_sub_0011_synced_video.mp4
⚡ Encoding 720p proxy (one-time)...
✅ Done in 198.7s · 46.7 MB


<IPython.core.display.Javascript object>

In [ ]:
# @title 📝 Rep Phase Annotator { display-mode: "form" }
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# ── Colab Form Parameters ───────────────────────────────────────────
annotator_id = 2  # @param {type:"integer"}

# ── inherited from the video player cell ───────────────────────────
if "entry_type" not in globals() or "target_entry_id" not in globals():
    raise RuntimeError("⚠️ Run the video player cell first — it defines entry_type and target_entry_id.")
_entry_type      = entry_type
_target_entry_id = target_entry_id
# ────────────────────────────────────────────────────────────────────

PHASES = [
    "start_rest", "setup", "first_pull", "second_pull", "clean",
    "middle_rest", "jerk_phase_one", "jerk_phase_two",
    "rest_up", "bring_down", "end_rest", "end_rep",
]

def new_rep_template():
    return {p: 0 for p in PHASES}

def tag_for(aid: int) -> str:
    return f"rep_segments_a{aid}"

state = {
    "annotator_id": annotator_id,
    "current_rep_idx": 0,
    "annotations": [],
    "is_locked": False,
    "is_updating_ui": False,  # <-- Flag to suppress unwanted input callbacks
}

# ── widgets ─────────────────────────────────────────────────────────
rep_selector = widgets.Dropdown(
    options=[("Rep 1", 0)],
    value=0,
    description="Active Rep:",
    layout=widgets.Layout(width="200px"),
)
btn_add_rep    = widgets.Button(description="➕ Add Rep",     button_style="success", layout=widgets.Layout(width="120px"))
btn_del_rep    = widgets.Button(description="🗑️ Delete Rep", button_style="danger",  layout=widgets.Layout(width="130px"))
btn_reload     = widgets.Button(description="🔄 Reload",                             layout=widgets.Layout(width="110px"))
btn_save_store = widgets.Button(description="💾 Save to Store", button_style="primary", layout=widgets.Layout(width="160px"))
btn_unlock     = widgets.Button(description="🔓 Unlock to Edit", button_style="warning", layout=widgets.Layout(width="160px"))
comment_input  = widgets.Text(value="", placeholder="Optional comment", description="Comment:", layout=widgets.Layout(width="300px"))
status_label   = widgets.HTML(value="")

phase_inputs, phase_rows = {}, []
for phase in PHASES:
    int_box = widgets.IntText(
        value=0,
        description=phase,
        style={"description_width": "140px"},
        layout=widgets.Layout(width="340px"),
    )
    def _make_cb(p):
        def _cb(change):
            # Block callback if UI is updating programmatically
            if state["is_updating_ui"]:
                return
            idx = state["current_rep_idx"]
            if 0 <= idx < len(state["annotations"]):
                state["annotations"][idx][p] = int(change["new"] or 0)
                refresh_preview()
        return _cb
    int_box.observe(_make_cb(phase), names="value")
    phase_inputs[phase] = int_box
    phase_rows.append(int_box)

json_preview = widgets.Textarea(
    value="",
    description="",
    layout=widgets.Layout(width="460px", height="460px"),
)

# ── helpers ─────────────────────────────────────────────────────────
def refresh_preview():
    json_preview.value = json.dumps(state["annotations"], indent=2)

def set_ui_locked(locked: bool):
    """Enable or disable interactive widgets and toggle the view."""
    state["is_locked"] = locked
    for box in phase_inputs.values():
        box.disabled = locked
    rep_selector.disabled = locked
    btn_add_rep.disabled = locked
    btn_del_rep.disabled = locked
    btn_reload.disabled = locked
    btn_save_store.disabled = locked
    comment_input.disabled = locked
    json_preview.disabled = locked

    editor_container.layout.display = "none" if locked else "flex"
    lock_banner.layout.display = "flex" if locked else "none"

def switch_to_rep(idx: int):
    reps = state["annotations"]
    if not reps:
        return
    idx = max(0, min(idx, len(reps) - 1))
    state["current_rep_idx"] = idx

    # Suppress event firing while syncing values
    state["is_updating_ui"] = True
    try:
        rep_selector.unobserve(on_rep_selected, names="value")
        try:
            rep_selector.options = [(f"Rep {i+1}", i) for i in range(len(reps))]
            rep_selector.value = idx
        finally:
            rep_selector.observe(on_rep_selected, names="value")

        data = reps[idx]
        for p in PHASES:
            phase_inputs[p].value = int(data.get(p, 0))
    finally:
        state["is_updating_ui"] = False

    refresh_preview()

def load_from_store():
    aid = state["annotator_id"]
    tag = tag_for(aid)
    try:
        df = store.get_entry_processed_data(
            entry_type=_entry_type, entry_id=_target_entry_id, tag=tag,
        )
    except Exception:
        df = None

    if df is not None and not df.empty:
        cols = [c for c in PHASES if c in df.columns]
        state["annotations"] = df[cols].to_dict(orient="records")
        status_label.value = (
            f"<span style='color:#38bdf8;'>📂 Loaded {len(state['annotations'])} rep(s) "
            f"for Experiment <b>{_target_entry_id}</b> (tag <code>{tag}</code>).</span>"
        )
    else:
        state["annotations"] = [new_rep_template()]
        status_label.value = (
            f"<span style='color:#a1a1aa;'>📝 No prior data for <b>{_target_entry_id}</b>. "
            f"Initialized 1 blank rep.</span>"
        )

    switch_to_rep(0)

# ── callbacks with loading state ────────────────────────────────────
def on_rep_selected(change):
    if change["new"] is not None and not state["is_updating_ui"]:
        switch_to_rep(change["new"])

def on_add_rep(btn):
    orig_desc = btn.description
    btn.disabled = True
    btn.description = "⏳ Adding..."
    try:
        state["annotations"].append(new_rep_template())
        new_idx = len(state["annotations"]) - 1
        switch_to_rep(new_idx)
        status_label.value = f"<span style='color:#38bdf8;'>➕ Added and switched to <b>Rep {new_idx+1}</b>.</span>"
    finally:
        btn.description = orig_desc
        btn.disabled = False

def on_del_rep(btn):
    if len(state["annotations"]) <= 1:
        status_label.value = "<span style='color:#fbbf24;'>⚠️ At least 1 rep must remain.</span>"
        return
    orig_desc = btn.description
    btn.disabled = True
    btn.description = "⏳ Deleting..."
    try:
        removed_idx = state["current_rep_idx"]
        state["annotations"].pop(removed_idx)
        new_idx = max(0, removed_idx - 1)
        switch_to_rep(new_idx)
        status_label.value = f"<span style='color:#f87171;'>🗑️ Removed Rep {removed_idx+1}. Switched to <b>Rep {new_idx+1}</b>.</span>"
    finally:
        btn.description = orig_desc
        btn.disabled = False

def on_reload(btn):
    orig_desc = btn.description
    btn.disabled = True
    btn.description = "⏳ Reloading..."
    try:
        load_from_store()
        status_label.value = "<span style='color:#38bdf8;'>🔄 Reloaded from store (unsaved edits discarded).</span>"
    finally:
        btn.description = orig_desc
        btn.disabled = False

def on_unlock(btn):
    orig_desc = btn.description
    btn.disabled = True
    btn.description = "⏳ Unlocking..."
    try:
        set_ui_locked(False)
        status_label.value = "<span style='color:#fbbf24;'>🔓 Editor unlocked for modification.</span>"
    finally:
        btn.description = orig_desc
        btn.disabled = False

def on_save(btn):
    orig_desc = btn.description
    btn.disabled = True
    btn.description = "⏳ Saving..."
    status_label.value = "<span style='color:#94a3b8;'>⏳ Writing annotations to store...</span>"

    aid = state["annotator_id"]

    try:
        parsed = json.loads(json_preview.value)
        if isinstance(parsed, list) and parsed:
            state["annotations"] = parsed
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Invalid JSON: {e}</span>"
        btn.description = orig_desc
        btn.disabled = False
        return

    rows = []
    for i, rep in enumerate(state["annotations"]):
        row = {"annotator_id": aid, "rep": i + 1}
        for p in PHASES:
            row[p] = int(rep.get(p, 0))
        rows.append(row)
    df  = pd.DataFrame(rows)[["annotator_id", "rep"] + PHASES]
    tag = tag_for(aid)

    try:
        existing = store.get_entry_processed_data(
            entry_type=_entry_type, entry_id=_target_entry_id, tag=tag,
        )
    except Exception:
        existing = None

    try:
        if existing is not None and not existing.empty:
            store.replace_processed_data(_entry_type, _target_entry_id, tag, df)
            verb = "Replaced"
        else:
            store.add_processed_data(
                entry_type=_entry_type, entry_id=_target_entry_id,
                df=df, tag=tag,
                comment=comment_input.value or f"annotator={aid}",
            )
            verb = "Saved"

        lock_summary.value = (
            f"<div style='font-size:14px; color:#f4f4f5;'>"
            f"✅ <b>Successfully {verb}!</b> Saved <b>{len(df)} rep(s)</b> for Experiment "
            f"<code style='color:#38bdf8; font-size:15px;'>{_target_entry_id}</code> "
            f"(Annotator #{aid} → <code>{tag}</code>).<br/>"
            f"<span style='color:#94a3b8; font-size:12px; margin-top:4px; display:inline-block;'>"
            f"The editor is now locked to protect this data. To move to the next experiment, select it in the video cell and re-run.</span>"
            f"</div>"
        )
        set_ui_locked(True)
        status_label.value = ""
    except Exception as e:
        status_label.value = f"<span style='color:#f87171;'>❌ Store error: {e}</span>"
    finally:
        btn.description = orig_desc
        btn.disabled = False

# ── bindings ────────────────────────────────────────────────────────
rep_selector.observe(on_rep_selected, names="value")
btn_add_rep.on_click(on_add_rep)
btn_del_rep.on_click(on_del_rep)
btn_reload.on_click(on_reload)
btn_save_store.on_click(on_save)
btn_unlock.on_click(on_unlock)

# ── layout ──────────────────────────────────────────────────────────
header = widgets.HTML(
    f"<div style='display:flex; justify-content:space-between; align-items:center; "
    f"padding:10px 16px; background:#18181b; color:#f4f4f5; border-radius:8px; "
    f"margin-bottom:8px; border:1px solid #27272a; font-family:system-ui,sans-serif;'>"
    f"<div>"
    f"<span style='color:#94a3b8; font-size:11px; text-transform:uppercase; letter-spacing:0.5px;'>Target Experiment ID</span><br/>"
    f"<b style='color:#38bdf8; font-size:17px;'>🔬 {_target_entry_id}</b> "
    f"<span style='color:#71717a; font-size:12px; margin-left:8px;'>type: <code style='color:#fbbf24;'>{_entry_type}</code></span>"
    f"</div>"
    f"<div style='text-align:right;'>"
    f"<span style='color:#94a3b8; font-size:11px; text-transform:uppercase; letter-spacing:0.5px;'>Annotator</span><br/>"
    f"<b style='color:#4ade80; font-size:16px;'>ID #{annotator_id}</b>"
    f"</div>"
    f"</div>"
)

control_bar = widgets.HBox([
    rep_selector, btn_add_rep, btn_del_rep,
    widgets.HTML("<span style='border-left:1px solid #3f3f46; margin:0 8px; height:24px;'></span>"),
    comment_input, btn_save_store, btn_reload
], layout=widgets.Layout(align_items="center", margin="0 0 10px 0"))

left_panel = widgets.VBox([widgets.HTML("<b>Phase Frame Indices:</b>")] + phase_rows)
right_panel = widgets.VBox([
    widgets.HTML("<b>Live JSON Preview / Quick Edit:</b>"),
    json_preview,
])

editor_container = widgets.VBox([
    control_bar,
    widgets.HBox([left_panel, right_panel], layout=widgets.Layout(gap="24px")),
])

lock_summary = widgets.HTML(value="")
lock_banner = widgets.HBox([
    lock_summary,
    btn_unlock
], layout=widgets.Layout(
    display="none",
    justify_content="space-between",
    align_items="center",
    padding="16px",
    background="#1e293b",
    border="1px solid #38bdf8",
    border_radius="8px",
    margin="10px 0"
))

main_ui = widgets.VBox([
    header,
    status_label,
    lock_banner,
    editor_container,
])

# ── initial load ────────────────────────────────────────────────────
load_from_store()
display(main_ui)

✅ Processed data 'rep_segments_a13' replaced for entry 'complete_trials_0012'.
